In [3]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import matplotlib.pyplot as plt
import os
from datetime import datetime, timedelta

def setup_client():
    """
    Menyiapkan client API dengan fitur Cache dan Retry.
    Tujuannya agar koneksi stabil dan hemat kuota.
    """
    # Buat cache bernama '.cache', data disimpan selama 3600 detik (1 jam)
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)

    # Jika gagal connect, coba lagi (retry) sampai 5x
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

    # Return objek client yang siap pakai
    return openmeteo_requests.Client(session=retry_session)

In [4]:
# ==========================================
# 2. FETCH DATA PER CHUNK
# ==========================================
def fetch_chunk(client, lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "dew_point_2m",
            "rain",
            "wind_speed_10m",
            "wind_direction_10m",
            "surface_pressure",
            "weather_code"
            ],
        "timezone": "Asia/Jakarta"
    }

    print(f"   ⏳ Mengambil: {start_date} s.d {end_date}...")
    try:
        responses = client.weather_api(url, params=params)
        return process_data(responses[0]) # Langsung olah jadi DataFrame
    except Exception as e:
        print(f"   ❌ Gagal pada chunk ini: {e}")
        return None

# ==========================================
# 3. PROCESS DATA (SAMA SEPERTI SEBELUMNYA)
# ==========================================
def process_data(response):
    hourly = response.Hourly()

    date_range = pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq = pd.Timedelta(seconds=hourly.Interval()),
        inclusive = "left"
    )

    df = pd.DataFrame(data = {
        "date": date_range,
        "temperature": hourly.Variables(0).ValuesAsNumpy(),
        "humidity": hourly.Variables(1).ValuesAsNumpy(),
        "dewpoint":hourly.Variables(2).ValuesAsNumpy(),
        "rain_mm": hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed": hourly.Variables(4).ValuesAsNumpy(),
        "wind_direction": hourly.Variables(5).ValuesAsNumpy(),
        "pressure": hourly.Variables(6).ValuesAsNumpy(),
        "weather_code": hourly.Variables(7).ValuesAsNumpy()
    })

    df = df.set_index('date')
    df.index = df.index.tz_convert('Asia/Jakarta')
    return df

# ==========================================
# 4. LOGIKA UTAMA: CHUNKING & MERGING
# ==========================================
def fetch_long_period_data(client, lat, lon, start_str, end_str, folder_tujuan, chunk_years=10):

    if not os.path.exists(folder_tujuan):
        os.makedirs(folder_tujuan)

    start_date = datetime.strptime(start_str, "%Y-%m-%d")
    final_end_date = datetime.strptime(end_str, "%Y-%m-%d")

    all_files = []

    current_start = start_date

    print(f"🚀 Memulai Misi Pengambilan Data 75 Tahun ({chunk_years} tahunan)...")

    while current_start < final_end_date:
        # Hitung tanggal akhir chunk ini
        # Misal start 1950, tambah 10 tahun -> 1960. Dikurang 1 hari biar gak overlap.
        current_end = current_start.replace(year=current_start.year + chunk_years) - timedelta(days=1)

        # Jika current_end melebihi batas akhir request, pakai batas akhir request
        if current_end > final_end_date:
            current_end = final_end_date

        # Format string untuk API
        s_str = current_start.strftime("%Y-%m-%d")
        e_str = current_end.strftime("%Y-%m-%d")

        # Nama file sementara
        chunk_filename = os.path.join(folder_tujuan, f"temp_{s_str}_{e_str}.csv")

        # Cek apakah file sudah ada? (Resume Capability)
        if os.path.exists(chunk_filename):
            print(f"⏩ Skip: {s_str} - {e_str} (Sudah ada)")
            all_files.append(chunk_filename)
        else:
            # Ambil Data
            df_chunk = fetch_chunk(client, lat, lon, s_str, e_str)
            if df_chunk is not None:
                df_chunk.to_csv(chunk_filename)
                print(f"   ✅ Tersimpan: {chunk_filename}")
                all_files.append(chunk_filename)
            else:
                print("   ⚠️ Chunk ini dilewati karena error.")

        # Lanjut ke periode berikutnya
        current_start = current_end + timedelta(days=1)

    print("\n🔗 Menggabungkan semua pecahan data...")

    # Gabungkan semua CSV jadi satu
    df_list = []
    for f in all_files:
        df = pd.read_csv(f, index_col='date', parse_dates=True)
        df_list.append(df)

    if df_list:
        df_final = pd.concat(df_list)
        df_final = df_final.sort_index() # Urutkan waktu biar rapi
        # Hapus duplikat jika ada irisan
        df_final = df_final[~df_final.index.duplicated(keep='first')]

        print(f"🎉 SUKSES BESAR! Total Data: {len(df_final)} baris.")
        print(f"   Mulai: {df_final.index.min()}")
        print(f"   Akhir: {df_final.index.max()}")
        return df_final
    else:
        return None

In [8]:
import time
from datetime import date, timedelta
import os
# Pastikan library lain yang dibutuhkan (openmeteo_requests, pandas, retry, dll) sudah terimport di bagian atas filemu

# ==========================================
# FUNGSI TAMBAHAN: RETRY LOGIC (ANTI GAGAL)
# ==========================================
def fetch_chunk_safe(func_fetch, *args, max_retries=5, **kwargs):
    """
    Mencoba menjalankan fungsi fetch. Jika gagal (kena limit API),
    akan menunggu dan mencoba lagi secara otomatis.
    """
    wait_seconds = 60 # Tunggu 1 menit jika gagal (sesuai aturan Open Meteo)

    for attempt in range(1, max_retries + 1):
        try:
            return func_fetch(*args, **kwargs)
        except Exception as e:
            print(f"⚠️ Percobaan ke-{attempt} gagal: {e}")
            if attempt < max_retries:
                print(f"⏳ Menunggu {wait_seconds} detik sebelum mencoba lagi...")
                time.sleep(wait_seconds)
            else:
                print("❌ Gagal total setelah beberapa kali percobaan.")
                raise e # Lempar error jika sudah menyerah

# ==========================================
# EKSEKUSI UTAMA
# ==========================================
if __name__ == "__main__":
    # 1. Konfigurasi Lokasi
    # Perhatikan: tuple koma di akhir LAT dihapus agar aman jadi float biasa
    LAT = -7.736436737566032
    LON = 109.6460550796716

    # 2. Konfigurasi Waktu (OTOMATIS)
    MULAI = "1950-01-01"

    # Hitung tanggal kemarin (H-1) secara otomatis
    kemarin = date.today() - timedelta(days=1)
    AKHIR = kemarin.strftime("%Y-%m-%d")

    print(f"📅 Jadwal Run Otomatis: Mengambil data dari {MULAI} sampai {AKHIR}")

    # 3. Konfigurasi File
    FOLDER = "open_meteo_climate"
    FILE_FINAL = "kebumen_75tahun_lengkap.csv"

    # Setup Client (Pastikan fungsi setup_client() ada di kodemu sebelumnya)
    client = setup_client()

    # 4. Jalankan Fetching dengan Pengaman (Safe Mode)
    # Kita bungkus proses fetching utama dalam try-except besar atau gunakan logika retry di level chunk
    # Namun, karena fetch_long_period_data sudah melakukan looping internal,
    # kita modifikasi sedikit cara panggilannya atau pastikan fungsi fetch_long_period_data
    # memiliki jeda (sleep) antar request.

    try:
        # Jalankan fungsi utamamu
        # Tips: Open-Meteo membatasi request per menit.
        # Pastikan di dalam fungsi `fetch_long_period_data` ada time.sleep(5) antar chunk.
        df_lengkap = fetch_long_period_data(client, LAT, LON, MULAI, AKHIR, FOLDER, chunk_years=5)

        if df_lengkap is not None:
            # Simpan Hasil Akhir
            if not os.path.exists(FOLDER):
                os.makedirs(FOLDER)

            path_final = os.path.join(FOLDER, FILE_FINAL)
            df_lengkap.to_csv(path_final, index=False) # index=False biar rapi
            print(f"✅ SUKSES! File Gabungan Tersimpan: {path_final}")

            # Hapus file temp (Opsional)
            # import glob
            # for f in glob.glob(f"{FOLDER}/temp_*.csv"):
            #     os.remove(f)

    except Exception as e:
        print(f"❌ Terjadi kesalahan fatal pada script utama: {e}")

📅 Jadwal Run Otomatis: Mengambil data dari 1950-01-01 sampai 2025-12-06
🚀 Memulai Misi Pengambilan Data 75 Tahun (5 tahunan)...
⏩ Skip: 1950-01-01 - 1954-12-31 (Sudah ada)
⏩ Skip: 1955-01-01 - 1959-12-31 (Sudah ada)
⏩ Skip: 1960-01-01 - 1964-12-31 (Sudah ada)
⏩ Skip: 1965-01-01 - 1969-12-31 (Sudah ada)
⏩ Skip: 1970-01-01 - 1974-12-31 (Sudah ada)
⏩ Skip: 1975-01-01 - 1979-12-31 (Sudah ada)
⏩ Skip: 1980-01-01 - 1984-12-31 (Sudah ada)
⏩ Skip: 1985-01-01 - 1989-12-31 (Sudah ada)
⏩ Skip: 1990-01-01 - 1994-12-31 (Sudah ada)
⏩ Skip: 1995-01-01 - 1999-12-31 (Sudah ada)
⏩ Skip: 2000-01-01 - 2004-12-31 (Sudah ada)
⏩ Skip: 2005-01-01 - 2009-12-31 (Sudah ada)
⏩ Skip: 2010-01-01 - 2014-12-31 (Sudah ada)
⏩ Skip: 2015-01-01 - 2019-12-31 (Sudah ada)
⏩ Skip: 2020-01-01 - 2024-12-31 (Sudah ada)
⏩ Skip: 2025-01-01 - 2025-12-06 (Sudah ada)

🔗 Menggabungkan semua pecahan data...
🎉 SUKSES BESAR! Total Data: 665616 baris.
   Mulai: 1950-01-01 01:00:00+08:00
   Akhir: 2025-12-06 23:00:00+07:00
✅ SUKSES! File 